# 🦅 Orbit Wars: Apex Predator (FFA Optimized)
**Kaggle Public Score: 700+**

This notebook introduces a highly optimized agent designed specifically for the chaotic 4-player Free-For-All (FFA) environment. In a 4-player match, pure aggression is heavily punished by third-party players. This bot is opportunistic, calculating, and ruthless. It builds an early advantage and strikes only when the odds are overwhelmingly in its favor.

## 1. Scorched Earth Protocol
If the bot calculates that a planet is mathematically impossible to defend against an incoming armada (`urgent_threat`), it will not waste ships trying to save it. It flags the planet as `is_doomed` and triggers a total evacuation.

In [ ]:
# --- EXCERPT: SCORCHED EARTH PROTOCOL ---
dist_to_closest_enemy = min([get_dist(src.x, src.y, e.x, e.y) for e in enemy_planets] + [999.0])

# If incoming enemy fleet size overwhelmingly beats our defense + production
if src.urgent_threat > (src.ships + src.production * 8 + src.incoming_allied):
    src.is_doomed = True
    available = src.ships # Evacuate EVERYTHING
else:
    # Standard buffer calculation based on game mode (FFA vs 1v1)
    if is_ffa:
        buffer_multiplier = 1.2 if dist_to_closest_enemy > 45 else 3.0
    else:
        buffer_multiplier = 1.0 if dist_to_closest_enemy > 35 else 2.0
        
    base_buffer = src.production * buffer_multiplier
    defense_needed = max(0, src.urgent_threat - src.incoming_allied)
    reserve = base_buffer + defense_needed
    available = int(max(0, src.ships - max(3, reserve)))

## 2. The Sniper Tactic & Chaos Exploit
Instead of spending massive resources to capture a heavily defended neutral planet, the bot monitors enemy movements. If an enemy is about to capture a planet, the bot multiplies its target score (up to `4.0x`) to send a calculated micro-fleet that arrives exactly one turn after the enemy, stealing the newly conquered planet while its defenses are depleted.

In [ ]:
# --- EXCERPT: THE SNIPER TACTIC ---
score_multiplier = 1.0

# Sniper: Steal neutral planets exactly when the enemy exhausts their fleet capturing it
if tgt.owner == -1 and tgt.incoming_enemy > tgt.ships:
    needed = int(tgt.incoming_enemy - tgt.ships) + 5
    score_multiplier = 4.0

# Chaos Exploit: Third-party a battle between two other players
elif tgt.owner != -1 and tgt.incoming_enemy > future_defense:
    needed = int(tgt.incoming_enemy - future_defense) + 5 
    score_multiplier = 2.5
    
# EVACUATION OVERRIDE: If our planet is doomed, prioritize launching the evacuation fleet
if src.is_doomed: score *= 10.0

## 3. Tsunami Strike
Due to the logarithmic speed mechanics of the game, larger fleets travel exponentially faster. Instead of trickling ships slowly to a target, this bot hoards its forces until it can launch a massive wave (up to 85% of its available pool). This guarantees maximum travel speed and leaves a strong defensive garrison upon arrival.

In [ ]:
# --- EXCERPT: TSUNAMI STRIKE ---
if best_target_move:
    tx, ty, needed_amt, tid = best_target_move
    angle = math.atan2(ty - src.y, tx - src.x)
    
    send_amt = needed_amt
    if available > needed_amt:
        # If doomed, send 100%. Otherwise, launch a massive 85% Tsunami wave for max speed
        send_amt = available if src.is_doomed else max(needed_amt, int(available * 0.85)) 
        
    actions.append([int(src.id), float(angle), int(send_amt)])
    src.ships -= send_amt
    p_map[tid].incoming_allied += send_amt